# Кредитный скоринг (Альфа) — победный пайплайн
Прогон сверху вниз даёт `submission_final_blend.csv` — rank-average четырёх членов: Hybrid, Pretrain(5-fold), NTP, NTP2(5-fold).

## Настройка и загрузка данных

In [ ]:
import os, gc, warnings, itertools
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_style('whitegrid'); HAS_SNS = True
except Exception:
    HAS_SNS = False
plt.rcParams['figure.figsize'] = (10, 5)

class CFG:
    DATA_DIR     = '.'
    TRAIN_DATA   = 'train_data.parquet'
    TEST_DATA    = 'test_data.parquet'
    TRAIN_TARGET = 'train_target.csv'
    ID_COL = 'id'; SEQ_COL = 'rn'; TARGET = 'flag'
    SAMPLE_IDS = None     # None = все id; иначе — размер подвыборки id

path = lambda f: os.path.join(CFG.DATA_DIR, f)
print('Готово.')

In [ ]:
def reduce_mem(df):
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c].dtype):
            lo, hi = df[c].min(), df[c].max()
            if lo >= 0:
                df[c] = df[c].astype(np.uint8 if hi < 255 else np.uint16 if hi < 65535 else np.uint32)
            else:
                df[c] = df[c].astype(np.int8 if (lo > -128 and hi < 127) else np.int16 if (lo > -32768 and hi < 32767) else np.int32)
    return df

train  = reduce_mem(pd.read_parquet(path(CFG.TRAIN_DATA)))
target = pd.read_csv(path(CFG.TRAIN_TARGET))
print('train_data :', train.shape, '| строк-продуктов:', len(train))
print('target     :', target.shape, '| уникальных id:', target[CFG.ID_COL].nunique())
print('Память train:', round(train.memory_usage(deep=True).sum() / 1024**2, 1), 'МБ')
print(train.dtypes.value_counts().to_string())
train.head()

## Статические id-агрегаты

In [ ]:
# Фичи: агрегаты истории продуктов на уровне id
from tqdm.auto import tqdm

ID, RN, TGT = CFG.ID_COL, CFG.SEQ_COL, CFG.TARGET
feat_cols = [c for c in train.columns if c not in (ID, RN)]

counters = [c for c in ['pre_loans5','pre_loans530','pre_loans3060','pre_loans6090','pre_loans90'] if c in train.columns]
ratios   = [c for c in ['pre_util','pre_over2limit','pre_maxover2limit'] if c in train.columns]
sums_b   = [c for c in ['pre_loans_credit_limit','pre_loans_next_pay_summ','pre_loans_outstanding',
                        'pre_loans_max_overdue_sum','pre_loans_total_overdue','pre_loans_credit_cost_rate'] if c in train.columns]
dates_b  = [c for c in ['pre_since_opened','pre_since_confirmed','pre_pterm','pre_fterm',
                        'pre_till_pclose','pre_till_fclose'] if c in train.columns]
enc_cat  = [c for c in train.columns if c.startswith('enc_') and not c.startswith('enc_paym_')]
enc_pay  = sorted([c for c in train.columns if c.startswith('enc_paym_')])
flags    = sorted([c for c in train.columns if c.startswith('is_zero_')]) + \
           [c for c in ['pclose_flag','fclose_flag'] if c in train.columns]

# какие агрегаты считать для каждого признака
AGG = {}
for c in counters: AGG[c] = ['sum','max','mean']     # суммарная/худшая просрочка по истории
for c in ratios:   AGG[c] = ['mean','max','std']      # утилизация: уровень + разброс
for c in sums_b:   AGG[c] = ['mean','max','min']
for c in dates_b:  AGG[c] = ['min','max','mean']
for c in enc_cat:  AGG[c] = ['nunique','max']
for c in flags:    AGG[c] = ['mean','max']            # доля продуктов с флагом

def build_features(df):
    gb = df.groupby(ID, sort=True)
    parts = [gb.size().rename('n_prod').to_frame()]
    for c, funcs in tqdm(AGG.items(), desc='id-агрегаты', leave=False):
        a = gb[c].agg(funcs); a.columns = [f'{c}__{f}' for f in funcs]; parts.append(a)
    # платёжная лента enc_paym -> поведенческие признаки (средний/худший статус по продукту, затем по id)
    if enc_pay:
        rm = df[enc_pay].mean(axis=1); rx = df[enc_pay].max(axis=1)
        tmp = pd.DataFrame({ID: df[ID].values, 'pm': rm.values, 'px': rx.values})
        pg = tmp.groupby(ID)
        parts.append(pg.agg(paym_mean=('pm','mean'), paym_max=('px','max'), paym_std=('pm','std')))
    Xf = pd.concat(parts, axis=1)
    # снимок последнего продукта (свежайшее состояние) — отдельный сильный блок
    last = df.loc[gb[RN].idxmax()].set_index(ID)[feat_cols]
    last.columns = [f'{c}__last' for c in feat_cols]
    Xf = Xf.join(last)
    # тренды: свежее значение минус среднее по истории (динамика к последнему продукту)
    for c in ['pre_util','pre_loans_outstanding','pre_loans_credit_limit']:
        if f'{c}__last' in Xf.columns and f'{c}__mean' in Xf.columns:
            Xf[f'{c}__trend'] = Xf[f'{c}__last'].astype('float32') - Xf[f'{c}__mean'].astype('float32')
    return Xf.replace([np.inf, -np.inf], np.nan).astype('float32').fillna(0)

X = build_features(train)
y = target.set_index(ID).loc[X.index, TGT].astype('uint8')
print('Матрица признаков:', X.shape, '| дефолтов %.2f%%' % (y.mean() * 100))
gc.collect()
X.head()

## Последовательности продуктов (CSR) + temporal split

In [ ]:
# Сборка последовательностей продуктов на id (CSR: плоская матрица + offsets)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_FEATS = feat_cols          # бакет-признаки
K_MAX = 20                     # последние K продуктов по rn (median=7, max=55)

def seqs_from_df(df):
    order = np.lexsort((df[RN].values, df[ID].values))         # сорт по (id, затем rn возр.)
    mat = df[SEQ_FEATS].to_numpy(np.int16)[order]              # [n_rows, F], бакеты как индексы
    ids = df[ID].values[order]
    uids, starts, counts = np.unique(ids, return_index=True, return_counts=True)
    return mat, starts.astype(np.int64), counts.astype(np.int64), uids

mat, starts, counts, uids = seqs_from_df(train)
cards = (mat.max(axis=0) + 1).astype(np.int64)                 # размер словаря на каждый признак
y_arr = target.set_index(ID).loc[uids, TGT].to_numpy(np.float32)

# temporal split: поздние 20% id в valid
cut = np.sort(uids)[int(len(uids) * 0.8)]
tr_pos = np.where(uids <= cut)[0]
va_pos = np.where(uids >  cut)[0]
print(f'seqs: {mat.shape} | id={len(uids)} | train={len(tr_pos)} valid={len(va_pos)} | cut id={cut}')
print('словари признаков: min', int(cards.min()), '| max', int(cards.max()),
      '| суммарная ширина эмбеддингов будет посчитана в 9.2')

In [ ]:
# Dataset и модель: эмбеддинги бакетов -> BiGRU -> attention-pooling -> голова
class SeqDataset(Dataset):
    def __init__(self, mat, starts, counts, labels, pos, k=K_MAX):
        self.mat, self.starts, self.counts = mat, starts, counts
        self.labels, self.pos, self.k = labels, pos, k
    def __len__(self):
        return len(self.pos)
    def __getitem__(self, i):
        j = self.pos[i]; s = self.starts[j]; c = self.counts[j]
        a = s + max(0, c - self.k)                       # последние k продуктов (по rn)
        x = torch.from_numpy(self.mat[a:s + c].astype(np.int64))
        return x, self.labels[j]

def collate(batch):
    seqs, ys = zip(*batch)
    lens = [s.shape[0] for s in seqs]; L = max(lens); F = seqs[0].shape[1]
    X = torch.zeros(len(seqs), L, F, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i, s in enumerate(seqs):
        X[i, :lens[i]] = s; mask[i, :lens[i]] = True     # right-pad, реальные шаги в начале
    return X, mask, torch.tensor(ys, dtype=torch.float32)

def emb_dim(card):
    return int(min(24, round(1.6 * card ** 0.56)))       # размер эмбеддинга от мощности словаря

class SeqClassifier(nn.Module):
    def __init__(self, cards, hidden=128, dropout=0.2):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(int(c), emb_dim(int(c))) for c in cards])
        D = sum(emb_dim(int(c)) for c in cards)
        self.inp_drop = nn.Dropout(dropout)
        self.gru  = nn.GRU(D, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(2 * hidden, 1)
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x, mask):
        h = torch.cat([emb(x[:, :, i]) for i, emb in enumerate(self.embs)], dim=-1)
        out, _ = self.gru(self.inp_drop(h))              # [B, L, 2H]
        score = self.attn(out).squeeze(-1).masked_fill(~mask, float('-inf'))
        w = torch.softmax(score, dim=1).unsqueeze(-1)    # внимание по времени, паддинг занулён
        pooled = (out * w).sum(1)                        # [B, 2H]
        return self.head(pooled).squeeze(-1)

print('SeqDataset / SeqClassifier определены.')

## Инференс-хелпер

In [ ]:
# Инференс-хелпер: rank-average по сидам
def predict_ens(models, mt, st, ct, n):
    ds = SeqDataset(mt, st, ct, np.zeros(n, np.float32), np.arange(n))
    dl = DataLoader(ds, batch_size=2048, shuffle=False, collate_fn=collate,
                    num_workers=globals().get('NUM_WORKERS', 0), pin_memory=True)
    ranks = []
    for net in models:
        net.eval(); P = []
        with torch.no_grad():
            for xb, m, _ in tqdm(dl, desc='infer', leave=False):
                with torch.amp.autocast('cuda'):
                    P.append(torch.sigmoid(net(xb.to(device), m.to(device))).float().cpu())
        ranks.append(pd.Series(torch.cat(P).numpy()).rank(pct=True).values)    # rank-avg по сидам
    return np.mean(ranks, axis=0)
print('predict_ens определён.')


## Hybrid: BiGRU + статические агрегаты → submission_hybrid_ens.csv

In [ ]:
# Статические признаки + Hybrid-датасет и модель
# Стандартизация статических признаков по train (tr_pos)
X_agg = X if isinstance(X, pd.DataFrame) else build_features(train)   # восстановление X при необходимости
static_cols = list(X_agg.columns)
Xstat = X_agg.reindex(uids).reindex(columns=static_cols).astype('float32')   # порядок как в uids
_mu = np.nanmean(Xstat.values[tr_pos], axis=0)
_sd = np.nanstd(Xstat.values[tr_pos], axis=0) + 1e-6
static_arr = np.nan_to_num((Xstat.values - _mu) / _sd, nan=0.0).astype('float32')
n_static = static_arr.shape[1]
print('static:', static_arr.shape, '| n_static =', n_static)

class HybridDataset(Dataset):
    def __init__(s, mat, starts, counts, labels, static, pos, k=K_MAX):
        s.mat, s.starts, s.counts, s.labels, s.static, s.pos, s.k = mat, starts, counts, labels, static, pos, k
    def __len__(s): return len(s.pos)
    def __getitem__(s, i):
        j = s.pos[i]; a = s.starts[j] + max(0, s.counts[j] - s.k); b = s.starts[j] + s.counts[j]
        return torch.from_numpy(s.mat[a:b].astype(np.int64)), s.static[j], s.labels[j]

def collate_h(batch):
    seqs, stats, ys = zip(*batch)
    lens = [x.shape[0] for x in seqs]; L = max(lens); F = seqs[0].shape[1]
    Xb = torch.zeros(len(seqs), L, F, dtype=torch.long); m = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i, x in enumerate(seqs): Xb[i, :lens[i]] = x; m[i, :lens[i]] = True
    return Xb, m, torch.from_numpy(np.stack(stats)), torch.tensor(ys, dtype=torch.float32)

class ChampNetHybrid(nn.Module):
    def __init__(self, cards, n_static, hidden=128, dropout=0.2, static_hidden=64):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(int(c), emb_dim(int(c))) for c in cards])
        D = sum(emb_dim(int(c)) for c in cards)
        self.gru = nn.GRU(D, hidden, 1, batch_first=True, bidirectional=True)
        self.H = 2 * hidden
        self.attn = nn.Linear(self.H, 1)
        self.static_enc = nn.Sequential(nn.Linear(n_static, static_hidden), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Sequential(nn.Linear(self.H * 3 + static_hidden, hidden), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x, mask, stat):
        B, L, _ = x.shape
        h = torch.cat([e(x[:, :, i]) for i, e in enumerate(self.embs)], -1)
        out, _ = self.gru(h)
        sc = self.attn(out).squeeze(-1).masked_fill(~mask, float('-inf'))
        ap = (out * torch.softmax(sc, 1).unsqueeze(-1)).sum(1)
        last = out[torch.arange(B, device=x.device), (mask.sum(1) - 1).clamp(min=0)]
        mx = out.masked_fill(~mask.unsqueeze(-1), float('-inf')).max(1).values
        pooled = torch.cat([ap, last, mx], -1)
        return self.head(torch.cat([pooled, self.static_enc(stat)], -1)).squeeze(-1)
print('HybridDataset / ChampNetHybrid определены.')

In [ ]:
# Обучение Hybrid: ансамбль 5 сидов (OneCycle + best-epoch)
SEEDS_H, EPOCHS, LR, BATCH = [42, 1, 2, 3, 4], 6, 2e-3, 1024
NUM_WORKERS = globals().get('NUM_WORKERS', 0)
def mk_loader_h(pos, bs, sh):
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0: kw['persistent_workers'] = True
    return DataLoader(HybridDataset(mat, starts, counts, y_arr, static_arr, pos), batch_size=bs,
                      shuffle=sh, collate_fn=collate_h, drop_last=sh, **kw)
pw = float((y_arr[tr_pos] == 0).sum()) / max(float((y_arr[tr_pos] == 1).sum()), 1.0)
rankavg = lambda preds: np.mean([pd.Series(p).rank(pct=True).values for p in preds], axis=0)

VALID_PRED = globals().get('VALID_PRED', {})
hyb_models, hyb_preds = [], []
for sd in SEEDS_H:
    torch.manual_seed(sd)
    net = ChampNetHybrid(cards, n_static).to(device)
    tl = mk_loader_h(tr_pos, BATCH, True); vl = mk_loader_h(va_pos, BATCH * 2, False)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(tl), pct_start=0.15)
    scaler = torch.amp.GradScaler('cuda')
    best, bp = 0.0, None
    for ep in range(EPOCHS):
        net.train()
        for Xb, m, S, yb in tqdm(tl, desc=f'hyb seed{sd} ep{ep+1}/{EPOCHS}', leave=False):
            Xb, m, S, yb = Xb.to(device, non_blocking=True), m.to(device, non_blocking=True), S.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda'): loss = bce(net(Xb, m, S), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        net.eval(); P = []
        with torch.no_grad():
            for Xb, m, S, _ in vl:
                with torch.amp.autocast('cuda'): P.append(torch.sigmoid(net(Xb.to(device), m.to(device), S.to(device))).float().cpu())
        pred = torch.cat(P).numpy(); a = roc_auc_score(y_arr[va_pos], pred)
        if a > best: best, bp = a, pred
        print(f'  hyb seed{sd} ep{ep+1}: AUC={a:.5f} (best {best:.5f})')
    hyb_models.append(net); hyb_preds.append(bp)
    VALID_PRED[f'Hybrid_s{sd}'] = pd.Series(bp, index=uids[va_pos])
    print(f'>> Hybrid ансамбль {len(hyb_preds)} сид(ов): AUC={roc_auc_score(y_arr[va_pos], rankavg(hyb_preds)):.5f}')
VALID_PRED['Hybrid_ensemble'] = pd.Series(rankavg(hyb_preds), index=uids[va_pos])

# опциональный бленд с Champ-ансамблем, если доступен
if 'Champ_ensemble' in VALID_PRED:
    both = pd.concat([VALID_PRED['Hybrid_ensemble'], VALID_PRED['Champ_ensemble']], axis=1).dropna()
    bl = both.rank(pct=True).mean(1)
    print(f'>> Hybrid_ens + Champ_ens: AUC={roc_auc_score(y_arr[va_pos], bl.loc[uids[va_pos]]):.5f}')

In [ ]:
# Инференс Hybrid-ансамбля на test
def predict_ens_h(models, mt, st, ct, stat_t, n):
    ds = HybridDataset(mt, st, ct, np.zeros(n, np.float32), stat_t, np.arange(n))
    dl = DataLoader(ds, batch_size=2048, shuffle=False, collate_fn=collate_h,
                    num_workers=globals().get('NUM_WORKERS', 0), pin_memory=True)
    ranks = []
    for net in models:
        net.eval(); P = []
        with torch.no_grad():
            for Xb, m, S, _ in tqdm(dl, desc='infer', leave=False):
                with torch.amp.autocast('cuda'):
                    P.append(torch.sigmoid(net(Xb.to(device), m.to(device), S.to(device))).float().cpu())
        ranks.append(pd.Series(torch.cat(P).numpy()).rank(pct=True).values)
    return np.mean(ranks, axis=0)

test = reduce_mem(pd.read_parquet(path(CFG.TEST_DATA)))
mt, st, ct, ut = seqs_from_df(test)
mt = np.minimum(mt, (cards - 1).astype(mt.dtype))
Xt = build_features(test).reindex(ut).reindex(columns=static_cols).astype('float32')   # те же агрегаты
stat_t = np.nan_to_num((Xt.values - _mu) / _sd, nan=0.0).astype('float32')              # train-статистики
pred = predict_ens_h(hyb_models, mt, st, ct, stat_t, len(ut))
sub = pd.DataFrame({ID: ut, TGT: pred})
sub.to_csv('submission_hybrid_ens.csv', index=False)
print('Сохранено submission_hybrid_ens.csv:', sub.shape, '| ср. ранг %.4f' % pred.mean())

## MLM-претрейн + 5-fold finetune → submission_pretrain_ens.csv

In [ ]:
# Энкодер + MLM-голова (претрейн) + FT-голова (классификатор)
class Encoder(nn.Module):
    def __init__(self, cards, hidden=128):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(int(c), emb_dim(int(c))) for c in cards])
        self.D = sum(emb_dim(int(c)) for c in cards)
        self.mask_vec = nn.Parameter(torch.zeros(self.D))      # обучаемый эмбеддинг "замаскировано"
        self.gru = nn.GRU(self.D, hidden, 1, batch_first=True, bidirectional=True)
        self.H = 2 * hidden
    def forward(self, x, mask, mask_pos=None):
        h = torch.cat([e(x[:, :, i]) for i, e in enumerate(self.embs)], -1)
        if mask_pos is not None:
            h = torch.where(mask_pos.unsqueeze(-1), self.mask_vec.to(h.dtype), h)
        return self.gru(h)[0]                                  # [B, L, 2H]

class MLMHead(nn.Module):
    def __init__(self, encoder, cards):
        super().__init__(); self.enc = encoder
        self.heads = nn.ModuleList([nn.Linear(encoder.H, int(c)) for c in cards])
    def loss(self, x, mask, mask_pos):
        out = self.enc(x, mask, mask_pos)
        sel = out[mask_pos]; tgt = x[mask_pos]                 # только замаскированные продукты
        if sel.shape[0] == 0: return None
        l = 0.0
        for f, head in enumerate(self.heads):
            l = l + torch.nn.functional.cross_entropy(head(sel), tgt[:, f])
        return l / len(self.heads)

class FTHead(nn.Module):
    def __init__(self, encoder, hidden=128, dropout=0.2):
        super().__init__(); self.enc = encoder; H = encoder.H
        self.attn = nn.Linear(H, 1)
        self.head = nn.Sequential(nn.Linear(H * 3, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x, mask):
        B = x.shape[0]; out = self.enc(x, mask)
        sc = self.attn(out).squeeze(-1).masked_fill(~mask, float('-inf'))
        ap = (out * torch.softmax(sc, 1).unsqueeze(-1)).sum(1)
        last = out[torch.arange(B, device=x.device), (mask.sum(1) - 1).clamp(min=0)]
        mx = out.masked_fill(~mask.unsqueeze(-1), float('-inf')).max(1).values
        return self.head(torch.cat([ap, last, mx], -1)).squeeze(-1)
print('Encoder / MLMHead / FTHead определены.')

In [ ]:
# Претрейн MLM на последовательностях (без меток)
import copy
PRE_EPOCHS, PRE_LR, BATCH, MASK_P = 8, 1e-3, 1024, 0.15   # гиперпараметры претрейна
NUM_WORKERS = globals().get('NUM_WORKERS', 0)
def mk_loader(pos, bs, sh):
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0: kw['persistent_workers'] = True
    return DataLoader(SeqDataset(mat, starts, counts, y_arr, pos), batch_size=bs,
                      shuffle=sh, collate_fn=collate, drop_last=sh, **kw)

torch.manual_seed(0)
encoder = Encoder(cards).to(device)
mlm = MLMHead(encoder, cards).to(device)
# пул претрейна: train + test последовательности (без меток)
from torch.utils.data import ConcatDataset
_testpre = reduce_mem(pd.read_parquet(path(CFG.TEST_DATA)))
_mt, _st, _ct, _ut = seqs_from_df(_testpre)
_mt = np.minimum(_mt, (cards - 1).astype(_mt.dtype))           # клип бакетов под train-словарь
ds_pre = ConcatDataset([SeqDataset(mat, starts, counts, y_arr, np.arange(len(uids))),
                        SeqDataset(_mt, _st, _ct, np.zeros(len(_ut), np.float32), np.arange(len(_ut)))])
_kw = dict(num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
if NUM_WORKERS > 0: _kw['persistent_workers'] = True
pl = DataLoader(ds_pre, batch_size=BATCH, shuffle=True, collate_fn=collate, **_kw)
print('пул претрейна (train+test):', len(ds_pre), 'последовательностей')
opt = torch.optim.AdamW(mlm.parameters(), lr=PRE_LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=PRE_LR, epochs=PRE_EPOCHS, steps_per_epoch=len(pl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')
for ep in range(PRE_EPOCHS):
    mlm.train(); tot, nb = 0.0, 0
    for xb, m, _ in tqdm(pl, desc=f'MLM ep{ep+1}/{PRE_EPOCHS}'):
        xb, m = xb.to(device, non_blocking=True), m.to(device, non_blocking=True)
        mp = m & (torch.rand_like(m, dtype=torch.float) < MASK_P)
        if mp.sum() == 0: continue
        opt.zero_grad()
        with torch.amp.autocast('cuda'): loss = mlm.loss(xb, m, mp)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); nb += 1
    print(f'  MLM ep{ep+1}: loss={tot/max(nb,1):.4f}')
pretrained_state = copy.deepcopy(encoder.state_dict())
print('Претрейн готов. Веса энкодера -> pretrained_state.')

In [ ]:
# 5-fold finetune Pretrain-трека (старт из pretrained_state)
from torch import nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
FT_EPOCHS, FT_LR, BATCH = 6, 2e-3, 1024
pw = float((y_arr == 0).sum()) / max(float((y_arr == 1).sum()), 1.0)
oof_pre = np.zeros(len(uids)); ft_kf = []
for fold, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(uids, y_arr.astype(int))):
    torch.manual_seed(42 + fold)
    enc = Encoder(cards).to(device); enc.load_state_dict(pretrained_state)   # старт из претрейна
    net = FTHead(enc).to(device)
    tl = mk_loader(tr, BATCH, True); vl = mk_loader(va, BATCH * 2, False)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
    opt = torch.optim.AdamW(net.parameters(), lr=FT_LR, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=FT_LR, epochs=FT_EPOCHS, steps_per_epoch=len(tl), pct_start=0.15)
    sc = torch.amp.GradScaler('cuda'); best, bp = 0.0, None
    for ep in range(FT_EPOCHS):
        net.train()
        for xb, m, yb in tqdm(tl, desc=f'pre-fold{fold} ep{ep+1}', leave=False):
            xb, m, yb = xb.to(device), m.to(device), yb.to(device); opt.zero_grad()
            with torch.amp.autocast('cuda'): loss = bce(net(xb, m), yb)
            sc.scale(loss).backward(); sc.step(opt); sc.update(); sch.step()
        net.eval(); P = []
        with torch.no_grad():
            for xb, m, _ in vl:
                with torch.amp.autocast('cuda'): P.append(torch.sigmoid(net(xb.to(device), m.to(device))).float().cpu())
        pr = torch.cat(P).numpy(); a = roc_auc_score(y_arr[va], pr)
        if a > best: best, bp = a, pr
    oof_pre[va] = bp; ft_kf.append(net); print(f'pre-fold{fold}: AUC={best:.5f}')
VALID_PRED['Pretrain_kfold'] = pd.Series(oof_pre, index=uids)
print('Pretrain 5-fold OOF на va_pos:', round(roc_auc_score(y_arr[va_pos], oof_pre[va_pos]), 5),
      )
# тест-предсказания: среднее 5 фолдов
test = reduce_mem(pd.read_parquet(path(CFG.TEST_DATA)))
mt, st, ct, ut = seqs_from_df(test); mt = np.minimum(mt, (cards - 1).astype(mt.dtype))
pd.DataFrame({ID: ut, TGT: predict_ens(ft_kf, mt, st, ct, len(ut))}).to_csv('submission_pretrain_ens.csv', index=False)
print('submission_pretrain_ens.csv сохранён')


## NTP causal-Transformer → submission_ntp_ens.csv

In [ ]:
# Causal-Transformer энкодер + NTP-голова (претрейн) + FT-голова
class TformerEncoder(nn.Module):
    def __init__(self, cards, d=128, nhead=4, layers=3, dropout=0.1):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(int(c), emb_dim(int(c))) for c in cards])
        D = sum(emb_dim(int(c)) for c in cards)
        self.proj = nn.Linear(D, d); self.pos = nn.Embedding(K_MAX, d); self.d = d
        lyr = nn.TransformerEncoderLayer(d, nhead, 4 * d, dropout, activation='gelu', batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(lyr, layers)
    def forward(self, x, mask, causal):
        B, L, _ = x.shape
        h = torch.cat([e(x[:, :, i]) for i, e in enumerate(self.embs)], -1)
        h = self.proj(h) + self.pos(torch.arange(L, device=x.device).clamp(max=K_MAX - 1))
        am = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), 1) if causal else None  # bool: True=запрещено
        return self.enc(h, mask=am, src_key_padding_mask=~mask)

class NTPHead(nn.Module):
    def __init__(self, encoder, cards):
        super().__init__(); self.enc = encoder
        self.heads = nn.ModuleList([nn.Linear(encoder.d, int(c)) for c in cards])
    def loss(self, x, mask):
        out = self.enc(x, mask, causal=True)
        pred = mask[:, 1:]                                  # позиция t валидна, если t+1 реальный
        h = out[:, :-1, :][pred]; tgt = x[:, 1:, :][pred]   # предсказываем следующий продукт
        if h.shape[0] == 0: return None
        l = 0.0
        for f, head in enumerate(self.heads):
            l = l + torch.nn.functional.cross_entropy(head(h), tgt[:, f])
        return l / len(self.heads)

class TFTHead(nn.Module):
    def __init__(self, encoder, hidden=128, dropout=0.2):
        super().__init__(); self.enc = encoder; H = encoder.d
        self.attn = nn.Linear(H, 1)
        self.head = nn.Sequential(nn.Linear(H * 3, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x, mask):
        B = x.shape[0]; out = self.enc(x, mask, causal=False)     # классификация — полное внимание
        sc = self.attn(out).squeeze(-1).masked_fill(~mask, float('-inf'))
        ap = (out * torch.softmax(sc, 1).unsqueeze(-1)).sum(1)
        last = out[torch.arange(B, device=x.device), (mask.sum(1) - 1).clamp(min=0)]
        mx = out.masked_fill(~mask.unsqueeze(-1), float('-inf')).max(1).values
        return self.head(torch.cat([ap, last, mx], -1)).squeeze(-1)
print('TformerEncoder / NTPHead / TFTHead определены.')

In [ ]:
# NTP-претрейн (causal) на train+test
import copy
from torch.utils.data import ConcatDataset
NTP_EPOCHS, NTP_LR, BATCH = 8, 5e-4, 1024
NUM_WORKERS = globals().get('NUM_WORKERS', 0)
def mk_loader(pos, bs, sh):
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0: kw['persistent_workers'] = True
    return DataLoader(SeqDataset(mat, starts, counts, y_arr, pos), batch_size=bs,
                      shuffle=sh, collate_fn=collate, drop_last=sh, **kw)

if '_mt' not in globals():                                   # переиспользуем test-последовательности, если есть
    _tp = reduce_mem(pd.read_parquet(path(CFG.TEST_DATA)))
    _mt, _st, _ct, _ut = seqs_from_df(_tp); _mt = np.minimum(_mt, (cards - 1).astype(_mt.dtype))
ds_pre = ConcatDataset([SeqDataset(mat, starts, counts, y_arr, np.arange(len(uids))),
                        SeqDataset(_mt, _st, _ct, np.zeros(len(_ut), np.float32), np.arange(len(_ut)))])
_kw = dict(num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
if NUM_WORKERS > 0: _kw['persistent_workers'] = True
pl = DataLoader(ds_pre, batch_size=BATCH, shuffle=True, collate_fn=collate, **_kw)
print('NTP пул (train+test):', len(ds_pre), 'последовательностей')

torch.manual_seed(0)
tenc = TformerEncoder(cards).to(device)
ntp = NTPHead(tenc, cards).to(device)
opt = torch.optim.AdamW(ntp.parameters(), lr=NTP_LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=NTP_LR, epochs=NTP_EPOCHS, steps_per_epoch=len(pl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')
for ep in range(NTP_EPOCHS):
    ntp.train(); tot, nb = 0.0, 0
    for xb, m, _ in tqdm(pl, desc=f'NTP ep{ep+1}/{NTP_EPOCHS}'):
        xb, m = xb.to(device, non_blocking=True), m.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast('cuda'): loss = ntp.loss(xb, m)
        if loss is None: continue
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); nb += 1
    print(f'  NTP ep{ep+1}: loss={tot/max(nb,1):.4f}')
ntp_state = copy.deepcopy(tenc.state_dict())
print('NTP-претрейн готов. Веса -> ntp_state.')

In [ ]:
# Fine-tune Transformer из NTP-претрейна: ансамбль сидов
from sklearn.metrics import roc_auc_score
FT_EPOCHS, FT_LR, SEEDS_T = 6, 5e-4, [42, 1, 2]
pw = float((y_arr[tr_pos] == 0).sum()) / max(float((y_arr[tr_pos] == 1).sum()), 1.0)
rankavg = lambda preds: np.mean([pd.Series(p).rank(pct=True).values for p in preds], axis=0)
VALID_PRED = globals().get('VALID_PRED', {})
ntp_models, ntp_preds = [], []
for sd in SEEDS_T:
    torch.manual_seed(sd)
    enc = TformerEncoder(cards).to(device); enc.load_state_dict(ntp_state)
    net = TFTHead(enc).to(device)
    tl = mk_loader(tr_pos, BATCH, True); vl = mk_loader(va_pos, BATCH * 2, False)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
    opt = torch.optim.AdamW(net.parameters(), lr=FT_LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=FT_LR, epochs=FT_EPOCHS, steps_per_epoch=len(tl), pct_start=0.15)
    scaler = torch.amp.GradScaler('cuda')
    best, bp = 0.0, None
    for ep in range(FT_EPOCHS):
        net.train()
        for xb, m, yb in tqdm(tl, desc=f'NTP-FT seed{sd} ep{ep+1}/{FT_EPOCHS}', leave=False):
            xb, m, yb = xb.to(device, non_blocking=True), m.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda'): loss = bce(net(xb, m), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        net.eval(); P = []
        with torch.no_grad():
            for xb, m, _ in vl:
                with torch.amp.autocast('cuda'): P.append(torch.sigmoid(net(xb.to(device), m.to(device))).float().cpu())
        pred = torch.cat(P).numpy(); a = roc_auc_score(y_arr[va_pos], pred)
        if a > best: best, bp = a, pred
        print(f'  NTP-FT seed{sd} ep{ep+1}: AUC={a:.5f} (best {best:.5f})')
    ntp_models.append(net); ntp_preds.append(bp)
    VALID_PRED[f'NTP_s{sd}'] = pd.Series(bp, index=uids[va_pos])
    print(f'>> NTP ансамбль {len(ntp_preds)}: AUC={roc_auc_score(y_arr[va_pos], rankavg(ntp_preds)):.5f}')
VALID_PRED['NTP_ensemble'] = pd.Series(rankavg(ntp_preds), index=uids[va_pos])
print('NTP-ансамбль готов.')

In [ ]:
# Инференс NTP-ансамбля на test
test = reduce_mem(pd.read_parquet(path(CFG.TEST_DATA)))
mt, st, ct, ut = seqs_from_df(test); mt = np.minimum(mt, (cards - 1).astype(mt.dtype))
pred = predict_ens(ntp_models, mt, st, ct, len(ut))
sub = pd.DataFrame({ID: ut, TGT: pred}); sub.to_csv('submission_ntp_ens.csv', index=False)
print('Сохранено submission_ntp_ens.csv:', sub.shape, '| ср. ранг %.4f' % pred.mean())

## NTP2 (d256/6L, recency+EMA) + 5-fold finetune → submission_ntp2_ens.csv

In [ ]:
# Масштабированный causal-Transformer (d256/6 слоёв) + recency
class TformerEncoder2(nn.Module):
    def __init__(self, cards, d=256, nhead=8, layers=6, dropout=0.1):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(int(c), emb_dim(int(c))) for c in cards])
        D = sum(emb_dim(int(c)) for c in cards)
        self.proj = nn.Linear(D, d)
        self.pos_start = nn.Embedding(K_MAX, d)              # позиция от начала (претрейн+finetune)
        self.pos_end = nn.Embedding(K_MAX, d)                # recency: от конца (только finetune)
        self.d = d
        lyr = nn.TransformerEncoderLayer(d, nhead, 4 * d, dropout, activation='gelu', batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(lyr, layers)
    def forward(self, x, mask, causal, recency=False):
        B, L, _ = x.shape
        h = torch.cat([e(x[:, :, i]) for i, e in enumerate(self.embs)], -1)
        ar = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.proj(h) + self.pos_start(ar.clamp(max=K_MAX - 1))
        if recency:
            dist = (mask.sum(1, keepdim=True) - 1 - ar).clamp(0, K_MAX - 1)
            h = h + self.pos_end(dist)
        am = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), 1) if causal else None
        return self.enc(h, mask=am, src_key_padding_mask=~mask)

class NTPHead2(nn.Module):
    def __init__(self, encoder, cards):
        super().__init__(); self.enc = encoder
        self.heads = nn.ModuleList([nn.Linear(encoder.d, int(c)) for c in cards])
    def loss(self, x, mask):
        out = self.enc(x, mask, causal=True, recency=False)
        pred = mask[:, 1:]; h = out[:, :-1, :][pred]; tgt = x[:, 1:, :][pred]
        if h.shape[0] == 0: return None
        l = 0.0
        for f, head in enumerate(self.heads): l = l + torch.nn.functional.cross_entropy(head(h), tgt[:, f])
        return l / len(self.heads)

class TFTHead2(nn.Module):
    def __init__(self, encoder, hidden=256, dropout=0.2):
        super().__init__(); self.enc = encoder; H = encoder.d
        self.attn = nn.Linear(H, 1)
        self.head = nn.Sequential(nn.Linear(H * 3, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x, mask):
        B = x.shape[0]; out = self.enc(x, mask, causal=False, recency=True)   # полное внимание + recency
        sc = self.attn(out).squeeze(-1).masked_fill(~mask, float('-inf'))
        ap = (out * torch.softmax(sc, 1).unsqueeze(-1)).sum(1)
        last = out[torch.arange(B, device=x.device), (mask.sum(1) - 1).clamp(min=0)]
        mx = out.masked_fill(~mask.unsqueeze(-1), float('-inf')).max(1).values
        return self.head(torch.cat([ap, last, mx], -1)).squeeze(-1)
print('TformerEncoder2 / NTPHead2 / TFTHead2 определены.')

In [ ]:
# Масштабированный NTP-претрейн на train+test
import copy
from torch.utils.data import ConcatDataset
PRE_EPOCHS, PRE_LR, BATCH = 12, 4e-4, 1024        # длиннее претрейн, ниже LR для большого трансформера
NUM_WORKERS = globals().get('NUM_WORKERS', 0)
def mk_loader(pos, bs, sh):
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0: kw['persistent_workers'] = True
    return DataLoader(SeqDataset(mat, starts, counts, y_arr, pos), batch_size=bs,
                      shuffle=sh, collate_fn=collate, drop_last=sh, **kw)
if '_mt' not in globals():
    _tp = reduce_mem(pd.read_parquet(path(CFG.TEST_DATA)))
    _mt, _st, _ct, _ut = seqs_from_df(_tp); _mt = np.minimum(_mt, (cards - 1).astype(_mt.dtype))
ds_pre = ConcatDataset([SeqDataset(mat, starts, counts, y_arr, np.arange(len(uids))),
                        SeqDataset(_mt, _st, _ct, np.zeros(len(_ut), np.float32), np.arange(len(_ut)))])
_kw = dict(num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
if NUM_WORKERS > 0: _kw['persistent_workers'] = True
pl = DataLoader(ds_pre, batch_size=BATCH, shuffle=True, collate_fn=collate, **_kw)
print('NTP2 пул (train+test):', len(ds_pre))

torch.manual_seed(0)
tenc2 = TformerEncoder2(cards).to(device)
print('параметров энкодера: %.1f млн' % (sum(p.numel() for p in tenc2.parameters()) / 1e6))
ntp2 = NTPHead2(tenc2, cards).to(device)
opt = torch.optim.AdamW(ntp2.parameters(), lr=PRE_LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=PRE_LR, epochs=PRE_EPOCHS, steps_per_epoch=len(pl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')
for ep in range(PRE_EPOCHS):
    ntp2.train(); tot, nb = 0.0, 0
    for xb, m, _ in tqdm(pl, desc=f'NTP2 ep{ep+1}/{PRE_EPOCHS}'):
        xb, m = xb.to(device, non_blocking=True), m.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast('cuda'): loss = ntp2.loss(xb, m)
        if loss is None: continue
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(ntp2.parameters(), 1.0)   # стабильность трансформера
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); nb += 1
    print(f'  NTP2 ep{ep+1}: loss={tot/max(nb,1):.4f}')
ntp2_state = copy.deepcopy(tenc2.state_dict())
print('NTP2-претрейн готов. Веса -> ntp2_state.')

In [ ]:
from sklearn.model_selection import StratifiedKFold
FT_EPOCHS, FT_LR, EMA_DECAY, BATCH = 6, 4e-4, 0.999, 1024
pw = float((y_arr==0).sum())/max(float((y_arr==1).sum()),1.0)
oof = np.zeros(len(uids)); ntp2_kf = []
for fold,(tr,va) in enumerate(StratifiedKFold(5,shuffle=True,random_state=42).split(uids, y_arr.astype(int))):
  torch.manual_seed(42+fold)
  enc = TformerEncoder2(cards).to(device); enc.load_state_dict(ntp2_state)   # старт из претрейна
  net = TFTHead2(enc).to(device)
  tl = mk_loader(tr,BATCH,True); vl = mk_loader(va,BATCH*2,False)
  bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
  opt = torch.optim.AdamW(net.parameters(),lr=FT_LR,weight_decay=1e-5)
  sch = torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=FT_LR,epochs=FT_EPOCHS,steps_per_epoch=len(tl),pct_start=0.15)
  sc = torch.amp.GradScaler('cuda'); ema=[p.detach().clone() for p in net.parameters()]
  best,bp,bema=0,None,None
  for ep in range(FT_EPOCHS):
      net.train()
      for xb,m,yb in tqdm(tl,desc=f'fold{fold} ep{ep+1}',leave=False):
          xb,m,yb=xb.to(device),m.to(device),yb.to(device); opt.zero_grad()
          with torch.amp.autocast('cuda'): loss=bce(net(xb,m),yb)
          sc.scale(loss).backward(); sc.step(opt); sc.update(); sch.step()
          with torch.no_grad():
              for e_,p in zip(ema,net.parameters()): e_.mul_(EMA_DECAY).add_(p.detach(),alpha=1-EMA_DECAY)
      bk=[p.detach().clone() for p in net.parameters()]
      for p,e_ in zip(net.parameters(),ema): p.data.copy_(e_)
      net.eval(); P=[]
      with torch.no_grad():
          for xb,m,_ in vl:
              with torch.amp.autocast('cuda'): P.append(torch.sigmoid(net(xb.to(device),m.to(device))).float().cpu())
      pred=torch.cat(P).numpy(); a=roc_auc_score(y_arr[va],pred)
      for p,b in zip(net.parameters(),bk): p.data.copy_(b)
      if a>best: best,bp,bema=a,pred,[e_.detach().clone() for e_ in ema]
  for p,e_ in zip(net.parameters(),bema): p.data.copy_(e_)
  oof[va]=bp; ntp2_kf.append(net); print(f'fold{fold}: AUC={best:.5f}')

VALID_PRED['NTP2_kfold'] = pd.Series(oof, index=uids)
print('NTP2 5-fold OOF AUC (все id):', round(roc_auc_score(y_arr, oof),5))
print('NTP2 5-fold OOF на va_pos:', round(roc_auc_score(y_arr[va_pos], oof[va_pos]),5))

# тест-предсказания NTP2: среднее 5 фолдов
test = reduce_mem(pd.read_parquet(path(CFG.TEST_DATA)))
mt, st, ct, ut = seqs_from_df(test); mt = np.minimum(mt, (cards - 1).astype(mt.dtype))
pd.DataFrame({ID: ut, TGT: predict_ens(ntp2_kf, mt, st, ct, len(ut))}).to_csv('submission_ntp2_ens.csv', index=False)
print('submission_ntp2_ens.csv сохранён (5-fold)')

## Финальный бленд → submission_final_blend.csv

In [ ]:
# Финальный бленд -> submission_final_blend.csv
FILES = ['submission_pretrain_ens.csv', 'submission_ntp_ens.csv', 'submission_ntp2_ens.csv', 'submission_hybrid_ens.csv']
ss = pd.read_csv(path('sample_submission.csv'))[['id']]
R = None
for f in FILES:
    r = pd.read_csv(path(f)).set_index('id')['flag'].rank(pct=True).rename(f)
    R = r.to_frame() if R is None else R.join(r)
blend = R.mean(axis=1).rename('flag')
out = ss.merge(blend.reset_index(), on='id', how='left')
assert out['flag'].notna().all() and len(out) == len(ss)
out.to_csv('submission_final_blend.csv', index=False)
print('submission_final_blend.csv:', out.shape, '| формат как sample_submission')
print(out.head(3).to_string(index=False))